# Lab 7: Large Language Models (LLMs)

In [ ]:
try:
    import transformers
except ImportError:
    %pip install -q transformers sentencepiece torch
    import transformers

from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForSeq2SeqLM

print("Libraries loaded successfully!")

## Task 1: Model Families

| # | Task | Most suitable family | Why |
| --- | --- | --- | --- |
| 1 | Sentiment classification | **Encoder-only** (BERT, RoBERTa) | reads the whole text and builds a representation to classify |
| 2 | Machine translation | **Encoder-decoder** (T5, BART) | the encoder reads the source sentence, the decoder writes the translation |
| 3 | Open-ended text generation | **Decoder-only** (GPT) | predicts the next token again and again |
| 4 | Named Entity Recognition | **Encoder-only** (BERT, RoBERTa) | labels every token using its full left and right context |
| 5 | Text summarization | **Encoder-decoder** (T5, BART) | the encoder reads the text, the decoder generates the shorter text |

## Task 2: Encoder-only Model (sentiment classification)

In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

sentences = ["The movie was excellent.",
             "The lecture was difficult to understand.",
             "I enjoyed learning about transformers."]

for sentence in sentences:
    result = classifier(sentence)[0]
    print(f"{sentence:<45} label: {result['label']:<9} score: {result['score']:.4f}")

## Task 3: Encoder-Decoder Model (summarization with FLAN-T5)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

paragraph = ("Large language models are neural networks trained on huge amounts of text to predict the next word. "
             "They are built on the transformer architecture, which uses attention to relate every word in a "
             "sentence to every other word. After pre-training, these models can be adapted to many tasks such as "
             "translation, summarization and question answering, often with just a short instruction. "
             "Their main limitations are the high cost of training and the fact that they can produce fluent "
             "but incorrect answers.")

prompt = "Summarize the following paragraph: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")      # text -> tokens
output = model.generate(**inputs, max_new_tokens=60)  # encoder reads, decoder generates

print("Paragraph:\n", paragraph)
print()
print("Summary:\n", tokenizer.decode(output[0], skip_special_tokens=True))

## Task 5: Short Reflection

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

A chatbot has to produce its reply one token at a time, and a decoder-only model is trained for exactly this: predicting the next token from all the previous tokens. 
Because of causal attention, every token only looks at what came before it, so the same model can read the conversation so far (the prompt) and simply continue it with the answer, with no separate encoder needed. 
Training only needs plain text with the next-word objective, so the model can be scaled to huge amounts of data and then fine-tuned to follow instructions and hold a conversation. 
Finally, the past computations can be cached, which makes generating long replies token by token fast and allows streaming.

## Optional Challenge: same prompt, different outputs

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")
prompt = "Natural language processing is"

print("WITH set_seed(42) before every run:")
for i in range(3):
    set_seed(42)
    out = generator(prompt, max_new_tokens=30, do_sample=True)
    print(f"  run {i + 1}: {out[0]['generated_text']}")

In [ ]:
print("WITHOUT set_seed (sampling cell run 3 times):")
for i in range(3):
    out = generator(prompt, max_new_tokens=30, do_sample=True)
    print(f"  run {i + 1}: {out[0]['generated_text']}")

**Question: why do we sometimes get different outputs for the same prompt?**

With `do_sample=True` the model does not always pick the most probable next token. It samples the next token randomly from the probability distribution over the vocabulary, and one different choice changes everything that follows. 
`set_seed(42)` starts the random number generator from the same state, so the same tokens are drawn and the output repeats. Greedy decoding (`do_sample=False`, as in Part A of the lab) has no randomness, so it always gives the same text.